In [1]:
!pip install tensorflow matplotlib

Defaulting to user installation because normal site-packages is not writeable


In [3]:
import zipfile
import os

# Extract zip file
with zipfile.ZipFile("archive (4) (1).zip", 'r') as zip_ref:
    zip_ref.extractall("dataset")

print("Extracted Successfully")

Extracted Successfully


In [4]:
for root, dirs, files in os.walk("dataset"):
    print(root)

dataset
dataset/cats_set
dataset/dogs_set


In [5]:
train_dir = "dataset/archive/train"
test_dir = "dataset/archive/test"

In [10]:
from tensorflow.keras.applications import MobileNetV2

# =========================
# DATASET PATH
# =========================
dataset_path = "dataset"

# =========================
# IMAGE SETTINGS
# =========================
img_height = 224
img_width = 224
batch_size = 32

# =========================
# LOAD DATASET
# =========================
train_ds = tf.keras.utils.image_dataset_from_directory(
    dataset_path,
    validation_split=0.2,
    subset="training",
    seed=42,
    image_size=(img_height, img_width),
    batch_size=batch_size
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    dataset_path,
    validation_split=0.2,
    subset="validation",
    seed=42,
    image_size=(img_height, img_width),
    batch_size=batch_size
)

# =========================
# PERFORMANCE OPTIMIZATION
# =========================
AUTOTUNE = tf.data.AUTOTUNE

train_ds = train_ds.prefetch(buffer_size=AUTOTUNE)

val_ds = val_ds.prefetch(buffer_size=AUTOTUNE)

# =========================
# DATA AUGMENTATION
# =========================
data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
])

# =========================
# LOAD PRETRAINED MODEL
# =========================
base_model = MobileNetV2(
    input_shape=(224,224,3),
    include_top=False,
    weights='imagenet'
)

# Freeze Base Model
base_model.trainable = False

# =========================
# BUILD MODEL
# =========================
model = keras.Sequential([

    data_augmentation,

    layers.Rescaling(1./255),

    base_model,

    layers.GlobalAveragePooling2D(),

    layers.Dense(128, activation='relu'),

    layers.Dropout(0.5),

    layers.Dense(1, activation='sigmoid')
])

# =========================
# COMPILE MODEL
# =========================
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

# =========================
# MODEL SUMMARY
# =========================
model.summary()

# =========================
# TRAIN MODEL
# =========================
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10
)

# =========================
# EVALUATE MODEL
# =========================
loss, accuracy = model.evaluate(val_ds)

print("\nValidation Accuracy:", accuracy)

Found 1000 files belonging to 2 classes.
Using 800 files for training.
Found 1000 files belonging to 2 classes.
Using 200 files for validation.
9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ sequential_2 (Sequential)       │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ rescaling_1 (Rescaling)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ ?                      │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,257,984 (8.61 MB)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 2,257,984 (8.61 MB)

Epoch 1/10
25/25 ━━━━━━━━━━━━━━━━━━━━ 7s 194ms/step - accuracy: 0.8838 - loss: 0.2516 - val_accuracy: 0.9850 - val_loss: 0.0385
Epoch 2/10
25/25 ━━━━━━━━━━━━━━━━━━━━ 4s 169ms/step - accuracy: 0.9613 - loss: 0.0949 - val_accuracy: 0.9800 - val_loss: 0.0470
Epoch 3/10
25/25 ━━━━━━━━━━━━━━━━━━━━ 4s 165ms/step - accuracy: 0.9663 - loss: 0.1033 - val_accuracy: 0.9800 - val_loss: 0.0411
Epoch 4/10
25/25 ━━━━━━━━━━━━━━━━━━━━ 4s 166ms/step - accuracy: 0.9550 - loss: 0.1167 - val_accuracy: 0.9750 - val_loss: 0.0457
Epoch 5/10
25/25 ━━━━━━━━━━━━━━━━━━━━ 4s 167ms/step - accuracy: 0.9762 - loss: 0.0642 - val_accuracy: 0.9750 - val_loss: 0.0401
Epoch 6/10
25/25 ━━━━━━━━━━━━━━━━━━━━ 4s 169ms/step - accuracy: 0.9712 - loss: 0.0793 - val_accuracy: 0.9750 - val_loss: 0.0506
Epoch 7/10
25/25 ━━━━━━━━━━━━━━━━━━━━ 4s 165ms/step - accuracy: 0.9800 - loss: 0.0680 - val_accuracy: 0.9700 - val_loss: 0.0806
Epoch 8/10
25/25 ━━━━━━━━━━━━━━━━━━━━ 4s 165ms/step - accuracy: 0.9762 - loss: 0.0626 - val_accuracy: 0.